以下の文の全ての組み合わせに対して、最終層の[CLS]トークンの埋め込みベクトルを用いてコサイン類似度を求めよ。
- “The movie was full of fun.”
- “The movie was full of excitement.”
- “The movie was full of crap.”
- “The movie was full of rubbish.”

In [5]:
from transformers import AutoTokenizer

model_id = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id)
def tokenize(text):
  inputs = tokenizer(text, return_tensors="pt")
  return inputs

In [6]:
text1 = "The movie was full of fun."
text2 = "The movie was full of excitement."
text3 = "The movie was full of crap."
text4 = "The movie was full of rubbish."
token1 = tokenize(text1)
token2 = tokenize(text2)
token3 = tokenize(text3)
token4 = tokenize(text4)
token1

{'input_ids': tensor([[ 101, 1996, 3185, 2001, 2440, 1997, 4569, 1012,  102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]])}

In [7]:
from transformers import AutoModel
import torch

model = AutoModel.from_pretrained(model_id)

def get_cls_embedding(inputs):
    with torch.no_grad():
        outputs = model(**inputs)
    # outputs.last_hidden_state の形状は [batch_size, sequence_length, hidden_size]
    # [CLS]トークンは常に先頭なので、インデックス [0, 0, :] で取り出す
    cls_emb = outputs.last_hidden_state[0, 0, :]
    return cls_emb

In [8]:
import torch.nn.functional as F

emb1 = get_cls_embedding(token1)
emb2 = get_cls_embedding(token2)
emb3 = get_cls_embedding(token3)
emb4 = get_cls_embedding(token4)

sentences = [text1, text2, text3, text4]
embeddings = [emb1, emb2, emb3, emb4]

for i in range(len(embeddings)):
    for j in range(i + 1, len(embeddings)):
        sim = F.cosine_similarity(embeddings[i], embeddings[j], dim=0)
        print(f"{sentences[i]}  vs  {sentences[j]}  →  {sim.item():.4f}")

The movie was full of fun.  vs  The movie was full of excitement.  →  0.9881
The movie was full of fun.  vs  The movie was full of crap.  →  0.9558
The movie was full of fun.  vs  The movie was full of rubbish.  →  0.9475
The movie was full of excitement.  vs  The movie was full of crap.  →  0.9541
The movie was full of excitement.  vs  The movie was full of rubbish.  →  0.9487
The movie was full of crap.  vs  The movie was full of rubbish.  →  0.9807
